# FinSafe — Kaggle setup and preflight
Organization: **Finsafe-team** · Account: **KeshavDixit11**

Enable Internet and an NVIDIA GPU in notebook settings. Add a Kaggle secret named
`HF_TOKEN` and enable it for this notebook. Accept Tiny Aya Global and Base access
conditions on Hugging Face first. Run cells in order. No training or publication
runs automatically. Keep this notebook private while setting up credentials.


In [ ]:
import sys, subprocess, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
    print("GPU available:", torch.cuda.is_available())
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(i, props.name, round(props.total_memory / 2**30, 1), "GiB")
except ImportError:
    print("PyTorch is not installed in this image.")
subprocess.run(["nvidia-smi"], check=False)


## Fetch the project
This only clones code. Training data and checkpoints are not downloaded.


In [ ]:
from pathlib import Path
project = Path("/kaggle/working/finsafe")
if not project.exists():
    subprocess.run(["git", "clone", "https://github.com/dixitkeshav/finsafe.git", str(project)], check=True)
else:
    print("Using existing checkout; no local changes will be overwritten.")
subprocess.run(["git", "-C", str(project), "rev-parse", "HEAD"], check=True)


## Prepare the core environment
The shared notebook kernel is left unchanged. A project-local Python 3.12
is used for curation and evaluation utilities. This does not install the GPU stack.


In [ ]:
import os
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
setup_env = os.environ.copy()
setup_env["UV_CACHE_DIR"] = "/kaggle/working/.cache/uv"
setup_env["UV_PYTHON_INSTALL_DIR"] = "/kaggle/working/.python"
subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.12", str(project / ".venv")], env=setup_env, check=True)
py = str(project / ".venv/bin/python")
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", py, "-r", "requirements-core.lock"], cwd=project, env=setup_env, check=True)
subprocess.run([sys.executable, "-m", "uv", "pip", "install", "--python", py, "--no-deps", "-e", "."], cwd=project, env=setup_env, check=True)


## Authenticate without displaying your token
The token needs access to the gated model repos and, later, write access to
Finsafe-team. Organization ownership is not verified by merely setting its name.


In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
run_env = setup_env.copy()
run_env["HF_TOKEN"] = hf_token
run_env["HF_HOME"] = "/kaggle/working/.cache/huggingface"
run_env["HF_NAMESPACE"] = "Finsafe-team"
run_env["TRL_ONLY_MODE"] = "1"
run_env["DISABLE_UNSLOTH_FOR_TRL"] = "1"
run_env["TOKENIZERS_PARALLELISM"] = "false"
# Only account/model metadata is printed; the token is never printed or saved.
auth_check = """from huggingface_hub import HfApi, hf_hub_download
api = HfApi()
identity = api.whoami()
print('Signed in as:', identity['name'])
print('Organizations:', [o['name'] for o in identity.get('orgs', [])])
for model in ['CohereLabs/tiny-aya-global', 'CohereLabs/tiny-aya-base']:
    sha = api.model_info(model).sha
    hf_hub_download(model, 'config.json', revision=sha)
    print('Access verified:', model, sha)
"""
subprocess.run([py, "-c", auth_check], env=run_env, check=True)


## Verify core code and the sample data workflow
These sample rows are only format checks, not research results.


In [ ]:
subprocess.run([str(project / ".venv/bin/finsafe"), "doctor"], cwd=project, env=run_env, check=True)
subprocess.run([py, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=project, env=run_env, check=True)
subprocess.run([str(project / ".venv/bin/finsafe"), "prepare", "--input", "data/examples/seed.jsonl", "--out", "data/processed"], cwd=project, env=run_env, check=True)
if not (project / "data/processed/curated").exists():
    subprocess.run([py, "scripts/curate.py"], cwd=project, env=run_env, check=True)
else:
    print("Preserving existing curation artifacts.")


## Next: choose the CUDA-compatible training environment
Save the first cell's Python, GPU, driver and PyTorch output and share it with the
project lead. Follow `docs/START_HERE.md` for the GPU environment. Do not blindly
install the Linux GPU lock: its CUDA build may not match this Kaggle host.

Once the GPU environment passes `finsafe doctor --gpu`, run five SFT steps before
any longer training. Use real, licensed data before measuring the hypothesis.
Save checkpoints and logs before the Kaggle session ends. HF artifact publishing
requires separate explicit commands and completed artifact cards.
